# Did Startups Rebrand as "AI" Companies After ChatGPT?

Haziq Ali · INFO 4617 Final Project

## 1. Collection

`collect.py` does the collecting. It reads the 2021 Forbes Cloud 100 from Forbes' own list data as the Wayback Machine archived it on August 10, 2021, asks the CDX Server API for every capture of each homepage from 2019 to 2025, picks the first working capture in each quarter, and downloads each one's raw HTML into `cache/`. Every request sends a User-Agent that identifies me and waits 5 seconds, and a 429 or 5xx answer waits before trying again. Run it from the repository's folder:

```bash
python collect.py
```

The cells below read what it saved.

In [1]:
import csv
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

CACHE = Path("cache")
DATA = Path("data")

companies = pd.read_csv(DATA / "companies.csv")
selected = pd.read_csv(CACHE / "selected.csv", dtype=str).fillna("")
log = pd.read_csv(CACHE / "fetch_log.csv", dtype=str).fillna("")
log = log.drop_duplicates(["slug", "timestamp"], keep="last")

print(len(companies), "companies,", (companies["homepage_source"] == "added").sum(), "homepages added by hand")
print(len(selected), "company-quarters,", (selected["timestamp"] != "").sum(), "with a capture")
print(len(log), "pages requested")
print(log["status"].value_counts().to_string())
print("first request:", log["fetched_at"].min(), "| last request:", log["fetched_at"].max())

100 companies, 48 homepages added by hand
2800 company-quarters, 2683 with a capture
2683 pages requested
status
200      2651
403        27
404         2
error       2
426         1
first request: 2026-11-08T19:39:27+00:00 | last request: 2026-11-09T06:02:17+00:00


## 2. Cleaning

Each page is parsed with BeautifulSoup. Scripts, styles and `noscript` blocks are removed, and what is left is the visible text, which includes the page title. AI terms are counted in that text: `AI` as a whole word in capitals, so that words such as "said" or "OpenAI" don't count, and "artificial intelligence", "machine learning" and "generative" in any case.

In [2]:
TERMS = {
    "ai_count": re.compile(r"\bAI\b"),
    "artificial_intelligence_count": re.compile(r"artificial intelligence", re.I),
    "machine_learning_count": re.compile(r"machine learning", re.I),
    "generative_count": re.compile(r"\bgenerative\b", re.I),
}

def clean_page(path):
    soup = BeautifulSoup(path.read_bytes(), "html.parser")
    lang = soup.html.get("lang", "") if soup.html else ""
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = " ".join(soup.get_text(" ").split())
    row = {"title": " ".join(title.split()), "html_lang": lang, "text_chars": len(text)}
    for name, pattern in TERMS.items():
        row[name] = len(pattern.findall(text))
    return row

pages = selected.merge(log[["slug", "timestamp", "status", "final_url"]], on=["slug", "timestamp"], how="left").fillna("")
rows = []
for _, s in pages.iterrows():
    row = {"slug": s["slug"], "quarter": s["quarter"], "capture_timestamp": s["timestamp"], "final_url": s["final_url"]}
    path = CACHE / "html" / s["slug"] / f"{s['timestamp']}.html"
    if not s["timestamp"]:
        row["status"] = "no_capture"
    elif not s["status"]:
        row["status"] = "not_fetched"
    elif s["status"] != "200" or not path.exists():
        row["status"] = "fetch_failed"
    else:
        row.update(clean_page(path))
        row["status"] = "thin_text" if row["text_chars"] < 200 else "ok"
    rows.append(row)

clean = pd.DataFrame(rows)
clean = companies[["rank", "company", "slug", "homepage"]].merge(clean, on="slug")
final = clean["final_url"].str.extract(r"/web/(\d{14})[^/]*/(?:https?://)?(?:www\.)?([^/:]+)")
clean["final_timestamp"] = final[0].fillna("")
clean["final_host"] = final[1].fillna("")
clean["redirected"] = (clean["final_host"] != "") & (clean["final_host"] != clean["homepage"].str.split("/").str[0])
clean["capture_date"] = pd.to_datetime(clean["final_timestamp"].where(clean["final_timestamp"] != "", clean["capture_timestamp"]).str[:8], format="%Y%m%d", errors="coerce").dt.date
counts = list(TERMS)
usable = clean["status"].isin(["ok", "thin_text"])
clean.loc[usable, counts] = clean.loc[usable, counts].astype(int)
clean["ai_terms_total"] = clean[counts].sum(axis=1, min_count=1)
clean["mentions_ai"] = (clean["ai_terms_total"] > 0).where(usable)

columns = ["rank", "company", "homepage", "quarter", "status", "capture_date", "final_host", "redirected", "html_lang", "title",
           "text_chars"] + counts + ["ai_terms_total", "mentions_ai"]
clean = clean[columns].sort_values(["rank", "quarter"]).reset_index(drop=True)
print(clean.shape)
print(clean["status"].value_counts().to_string())

(2800, 17)
status
ok              2587
no_capture       117
thin_text         64
fetch_failed      32


## 3. Checks

The proposal tested Notion's homepage by hand: 0 mentions in January 2022 and 14 in July 2025. Its rows here should tell the same story, and the rows with little text, usually pages built with JavaScript, need a look before they count as "no AI".

In [3]:
print(clean.loc[clean["company"] == "Notion", ["quarter", "status", "capture_date", "title", "ai_count", "artificial_intelligence_count", "ai_terms_total"]].dropna(subset=["title"]).to_string(index=False))
print()
print(clean.loc[clean["status"] == "thin_text", ["company", "quarter", "text_chars", "title"]].head(10).to_string(index=False))
print()
print(clean.loc[clean["html_lang"].str.startswith("zh", na=False), "company"].unique())
print()
ok_pages = clean[clean["status"] == "ok"].copy()
ok_pages["mentions_ai"] = ok_pages["mentions_ai"].astype(bool)
summary = ok_pages.groupby("quarter")["mentions_ai"].agg(["count", "mean"])
summary["mean"] = (summary["mean"] * 100).round(1)
print(summary.rename(columns={"count": "pages", "mean": "share mentioning AI (%)"}).to_string())

quarter    status capture_date                                                                          title  ai_count  artificial_intelligence_count  ai_terms_total
 2019Q1 thin_text   2019-01-13 Notion – The all-in-one workspace for your notes, tasks, wikis, and databases.       0.0                            0.0             0.0
 2019Q2 thin_text   2019-04-08 Notion – The all-in-one workspace for your notes, tasks, wikis, and databases.       0.0                            0.0             0.0
 2019Q3 thin_text   2019-07-02 Notion – The all-in-one workspace for your notes, tasks, wikis, and databases.       0.0                            0.0             0.0
 2019Q4 thin_text   2019-10-01 Notion – The all-in-one workspace for your notes, tasks, wikis, and databases.       0.0                            0.0             0.0
 2020Q1 thin_text   2020-01-02 Notion – The all-in-one workspace for your notes, tasks, wikis, and databases.       0.0                            0.0             0.

## 4. Saving

The full tidy dataset goes to `data/homepages_clean.csv`, which stays on my laptop. The repository gets a sample: every quarter for the ten highest-ranked companies.

In [4]:
clean.to_csv(DATA / "homepages_clean.csv", index=False)
sample = clean[clean["rank"] <= 10]
sample.to_csv(DATA / "sample_homepages_clean.csv", index=False)
print(len(clean), "rows saved;", len(sample), "rows in the sample")
print(sample[sample["status"] == "ok"].head(8)[["company", "quarter", "capture_date", "title", "ai_terms_total"]].to_string(index=False))

2800 rows saved; 280 rows in the sample
company quarter capture_date                                                      title  ai_terms_total
 Stripe  2019Q1   2019-01-01 Stripe - Online payment processing for internet businesses             1.0
 Stripe  2019Q2   2019-04-01 Online payment processing for internet businesses - Stripe             1.0
 Stripe  2019Q3   2019-07-03 Online payment processing for internet businesses - Stripe             1.0
 Stripe  2019Q4   2019-10-07 Online payment processing for internet businesses - Stripe             1.0
 Stripe  2020Q1   2020-01-01 Online payment processing for internet businesses - Stripe             1.0
 Stripe  2020Q2   2020-04-01 Online payment processing for internet businesses - Stripe             1.0
 Stripe  2020Q3   2020-07-01 Online payment processing for internet businesses - Stripe             0.0
 Stripe  2020Q4   2020-10-03 Online payment processing for internet businesses - Stripe             1.0
